### TOOLS
    - Models can request to call tools to perform specific task such as fethcing data from database ,searching web running code etc.It pair of:
    1. A schema including name of tools,a description of the tool and a argument/definition to call the tool.
    2. A function or coroutine to execute.

In [1]:
import os
from langchain_google_genai  import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=1.0,
    max_retries=2,   
)
# model.invoke("Tell about yourself")


In [12]:
from langchain.tools  import tool
@tool
def get_weather(city:str)->str:
    """Get the weather of city"""
    return f"The weather in {city} is rainy"
#binding a tool with llm
model_with_tools = model.bind_tools([get_weather])

In [10]:
response  =  model_with_tools.invoke("What is the weather of Himanchal pradesh?")
print(response)
for call in response.tool_calls:
    print (call['name'])
    print(call['args'])

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "Himachal Pradesh"}'}, '__gemini_function_call_thought_signatures__': {'TBY1brUm': 'EjQKMgEMOdbHTM6a3kAm3FIX2e4iQTU60MMXbRnbN9Obl2tm/CkiDr2kRhyFedMHxUd+e/hH'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019ed562-d323-7b13-90a3-a0db6d6d385c-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Himachal Pradesh'}, 'id': 'TBY1brUm', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 53, 'output_tokens': 18, 'total_tokens': 71, 'input_token_details': {'cache_read': 0}}
get_weather
{'city': 'Himachal Pradesh'}


### TOTAL EXECUTIONS LOOP

In [15]:
##Step 1: Model generate tool calls
messages = [{"role":"user","content":"What is the weather in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
## Step 2: Execute tool and collect results
for tool_calls in ai_msg.tool_calls:
    tool_res =  get_weather.invoke(tool_calls)
    messages.append(tool_res)
# Step 3: Pass result back to model for final response
# final_res = model_with_tools.invoke(messages)
print(tool_res.content)


The weather in New York is rainy


In [14]:
messages

[{'role': 'user', 'content': 'What is the weather in New York?'},
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "New York"}'}, '__gemini_function_call_thought_signatures__': {'Cu9719PE': 'EjQKMgEMOdbHva6oan7f6Pcw6HeJCJh/EY0ER96uMlxW6HugBArZGz81GWfLGOEdHW+Lemaj'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ed56b-c70e-7ae1-8989-42dd13113d1a-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'Cu9719PE', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 51, 'output_tokens': 17, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='The weather in New York is rainy', name='get_weather', tool_call_id='Cu9719PE')]